# BoxBunny — Master Runner Notebook

This notebook contains all commands needed to build, run, test, and manage the BoxBunny boxing training robot system.

**Demo Users:**
| User | Password | Level | Sessions | Rank |
|------|----------|-------|----------|------|
| alex | boxing123 | Beginner | 8 | Novice |
| maria | boxing123 | Intermediate | 35 | Fighter |
| jake | boxing123 | Advanced | 120 | Champion |
| sarah | coaching123 | Coach | 3 coaching | — |

---
## 0. Environment Setup
Run this cell first to set up the environment.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
# Source ROS 2 and workspace
echo "=== Environment Setup ==="
source /opt/ros/humble/setup.bash
if [ -f install/setup.bash ]; then
    source install/setup.bash
    echo "Workspace overlay sourced"
fi
echo "ROS_DISTRO: $ROS_DISTRO"
echo "Python: $(python3 --version)"
echo "Working dir: $(pwd)"
echo ""
echo "=== Package Status ==="
ros2 pkg list 2>/dev/null | grep boxbunny || echo "(packages not built yet — run Build cell first)"

---
## 1. Build Workspace
Builds all 4 ROS 2 packages: boxbunny_msgs, boxbunny_core, boxbunny_gui, boxbunny_dashboard

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash
echo "=== Building ROS 2 Workspace ==="
colcon build --symlink-install 2>&1
echo ""
echo "=== Build Complete ==="
source install/setup.bash
echo "Packages built:"
ros2 pkg list 2>/dev/null | grep boxbunny

---
## 2. Verify Messages
Check that all custom ROS 2 messages and services are built correctly.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Custom Messages ==="
ros2 interface list | grep boxbunny_msgs | head -30
echo ""
echo "=== Sample Message: ConfirmedPunch ==="
ros2 interface show boxbunny_msgs/msg/ConfirmedPunch
echo ""
echo "=== Sample Service: StartSession ==="
ros2 interface show boxbunny_msgs/srv/StartSession

---
## 3. Run Tests
Runs the full pytest suite (171 tests — no hardware needed).

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 -m pytest tests/ -v --tb=short 2>&1

---
## 4. Seed Demo Data
Creates demo users with realistic training history. Safe to re-run (idempotent).

In [ ]:
%%bash
set +e
python3 tools/demo_data_seeder.py

---
## 5. Launch System — Development Mode
Launches all ROS nodes + IMU simulator + GUI. Use this for development/testing.

**Note:** This launches in the background. Use the Stop cell below to shut down.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in dev mode..."
echo "(IMU simulator will open in a separate window)"
echo ""
ros2 launch boxbunny_core boxbunny_dev.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo "Run the 'Stop System' cell to shut down"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
echo ""
echo "=== Active ROS Nodes ==="
ros2 node list 2>/dev/null || echo "(nodes still starting...)"

---
## 6. Launch System — Full Production Mode
Launches everything including real hardware connections.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in FULL mode..."
ros2 launch boxbunny_core boxbunny_full.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
ros2 node list 2>/dev/null

---
## 7. Stop System
Shuts down all BoxBunny ROS nodes.

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
if [ -f /tmp/boxbunny_launch.pid ]; then
    PID=$(cat /tmp/boxbunny_launch.pid)
    kill $PID 2>/dev/null && echo "Stopped launch process $PID" || echo "Process already stopped"
fi
# Kill any remaining boxbunny nodes
pkill -f 'boxbunny_core' 2>/dev/null
pkill -f 'boxbunny_gui' 2>/dev/null
pkill -f 'boxbunny_dashboard' 2>/dev/null
pkill -f 'imu_simulator' 2>/dev/null
echo "All BoxBunny processes stopped"

---
## 8. Launch Individual Components
Run specific subsystems for isolated testing.

### 8a. IMU Simulator Only

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Starting IMU Simulator..."
python3 tools/imu_simulator.py &
sleep 2
echo "IMU Simulator running. Click pads to publish ROS topics."

### 8b. GUI Only

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
export QT_QPA_PLATFORM=xcb
echo "Starting BoxBunny GUI..."
python3 -m boxbunny_gui.app &
sleep 2
echo "GUI launched"

### 8c. Dashboard Server Only
Starts the dashboard with a **public tunnel URL** you can open on your phone from anywhere.
**Interrupt the cell (stop button) to shut down the server.**

In [ ]:
%run notebooks/scripts/dashboard_tunnel.py

### 8d. Headless Nodes Only (no GUI)

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching headless nodes..."
ros2 launch boxbunny_core headless.launch.py &
sleep 3
ros2 node list 2>/dev/null

---
## 9. Monitor ROS Topics
View live data flowing through the system.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Active ROS Topics ==="
ros2 topic list 2>/dev/null | grep boxbunny | sort
echo ""
echo "=== Active Nodes ==="
ros2 node list 2>/dev/null | grep -v _ros2cli | sort
echo ""
echo "=== Topic Hz (punch confirmed) ==="
timeout 3 ros2 topic hz /boxbunny/punch/confirmed 2>/dev/null || echo "(no data yet)"

### 9a. Echo a specific topic (5 messages)

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
# Change topic name as needed:
TOPIC="/boxbunny/imu/nav_event"
echo "Listening to $TOPIC (5 messages, 10s timeout)..."
timeout 10 ros2 topic echo $TOPIC --once 2>/dev/null || echo "(no messages received — is a publisher running?)"

---
## 10. Database Inspection
View demo user data in the databases.

In [ ]:
%run notebooks/scripts/inspect_main_db.py

In [ ]:
# Change username: alex, maria, jake
!python3 notebooks/scripts/inspect_user_db.py maria

---
## 11. Model Verification
Check that all ML models are present and loadable.

In [ ]:
%run notebooks/scripts/verify_models.py

---
## 12. LLM Test
Test the local LLM model (Qwen2.5-3B).

In [ ]:
%run notebooks/scripts/test_llm.py

---
## 13. Camera Test
Test the RealSense D435i camera connection.

In [ ]:
%run notebooks/scripts/test_camera.py

---
## 14. Action Prediction — Standalone Test
Test the CV model inference without ROS (uses camera directly).

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Running action prediction standalone test..."
echo "(Press Ctrl+C or close the window to stop)"
python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(camera not available — connect D435i to run this test)"

---
## 15. Dashboard Server Test
Start the phone dashboard and test API endpoints.

In [ ]:
%%bash
bash /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/test_dashboard_api.sh

---
## 16. Build Vue Frontend
Rebuild the phone dashboard Vue 3 SPA.

In [ ]:
%%bash
set +e
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
cd src/boxbunny_dashboard/frontend
echo "Node: $(node --version), npm: $(npm --version)"
echo ""
echo "=== Installing dependencies ==="
npm install 2>&1 | tail -3
echo ""
echo "=== Building Vue SPA ==="
npm run build 2>&1
echo ""
echo "=== Output ==="
ls -lh ../static/dist/ 2>/dev/null | head -5

---
## 17. Download Models
Download LLM model and check all model files.

In [ ]:
%%bash
set +e
bash scripts/download_models.sh

---
## 18. Full Setup (First Time)
One-command bootstrap for a fresh system. Installs deps, builds, downloads, seeds.

In [ ]:
%%bash
set +e
bash scripts/setup.sh 2>&1

---
## 19. Hardware Check
Quick status check of all hardware components.

In [ ]:
%run notebooks/scripts/hardware_check.py

---
## 20. Project Statistics

In [ ]:
%%bash
set +e
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
echo "=== BoxBunny Project Stats ==="
echo "Python files: $(find . -name '*.py' -not -path './_archive/*' -not -path './build/*' -not -path './install/*' -not -path '*/__pycache__/*' | wc -l)"
echo "Vue/JS files: $(find . \( -name '*.vue' -o -name '*.js' \) -not -path '*/node_modules/*' -not -path './_archive/*' -not -path './build/*' | wc -l)"
echo "ROS messages: $(find src/boxbunny_msgs -name '*.msg' | wc -l)"
echo "ROS services: $(find src/boxbunny_msgs -name '*.srv' | wc -l)"
echo "GUI widgets:  $(find src/boxbunny_gui/boxbunny_gui/widgets -name '*.py' -not -name '__init__.py' | wc -l)"
echo "GUI pages:    $(find src/boxbunny_gui/boxbunny_gui/pages -name '*.py' -not -name '__init__.py' | wc -l)"
echo "ROS nodes:    $(find src/boxbunny_core/boxbunny_core -name '*_node.py' -o -name '*_manager.py' -o -name '*_engine.py' -o -name '*_processor.py' | wc -l)"
echo "API endpoints: $(find src/boxbunny_dashboard/boxbunny_dashboard/api -name '*.py' -not -name '__init__.py' | wc -l)"
echo "Knowledge docs: $(find data/boxing_knowledge -type f | wc -l)"
echo "Test files:    $(find tests -name 'test_*.py' | wc -l)"
echo "Config files:  $(find config -type f | wc -l)"
echo "Launch files:  $(find src -name '*.launch.py' | wc -l)"

---
## 21. Sound Test
Play each sound effect to verify audio output. The first cell lists all available sounds; the second plays them all in sequence.

### 21a. List Available Sounds
Inventory of all `.wav` assets with file sizes.

In [ ]:
%run notebooks/scripts/list_sounds.py

### 21b. Play All Sounds in Sequence
Listen to every sound effect back-to-back with labels. Each sound plays as an embedded audio widget.

In [ ]:
%run notebooks/scripts/play_sounds.py

---
## 22. Dashboard with QR Code
Generate a scannable QR code for the phone dashboard, then start the server so you can connect from your phone.

**Prerequisite:** Run the *Seed Demo Data* cell (Section 4) first so users exist.

### 22a. Generate QR Code
Creates a QR code image that links to the dashboard URL on this device's local IP. Scan it with your phone (same Wi-Fi network required).

In [ ]:
%run notebooks/scripts/qr_local.py

### 22b. Start Dashboard Server
Launches the dashboard server in the background. After running this cell, scan the QR code above to access the dashboard from your phone.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

# Kill any existing dashboard process
pkill -f 'boxbunny_dashboard.server' 2>/dev/null
sleep 1

echo "=== Starting Dashboard Server ==="
python3 -m boxbunny_dashboard.server &
SERVER_PID=$!
echo $SERVER_PID > /tmp/boxbunny_dashboard.pid
sleep 3

# Verify it is running
if kill -0 $SERVER_PID 2>/dev/null; then
    echo "Dashboard server started (PID: $SERVER_PID)"
    echo ""
    echo "Open your phone browser and scan the QR code above, or visit:"
    echo "  http://$(hostname -I | awk '{print $1}'):8080"
    echo ""
    echo "To stop: run 'pkill -f boxbunny_dashboard.server' or use the Stop System cell"
else
    echo "ERROR: Dashboard server failed to start. Check logs above."
fi

---
## 23. Gesture Control Debug
Launch the gesture recognition node with a live camera debug window. This uses MediaPipe hand tracking to detect hand gestures for touchless GUI navigation.

**Requires:** RealSense D435i camera connected. Press Ctrl+C in the terminal (or interrupt the kernel) to stop.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== Gesture Control Debug Mode ==="
echo ""
echo "A camera window will open showing the live feed with hand landmarks."
echo ""
echo "Supported gestures:"
echo "  Open palm ........... Select / Enter"
echo "  Thumbs up ........... Confirm"
echo "  Peace sign .......... Go back"
echo "  Swipe left/right .... Navigate between options"
echo ""
echo "Hold each gesture for ~0.5 seconds to trigger the action."
echo "Press Ctrl+C or close the window to stop."
echo ""

ros2 run boxbunny_core gesture_node \
    --ros-args -p enabled:=true -p hold_duration_s:=0.5 2>&1 \
    || echo "(gesture node exited — is the camera connected?)"

---
## 24. IMU Simulator Visual Test
Launch the IMU pad simulator and monitor the nav events it publishes. Click the pad buttons in the simulator window to simulate punches and verify ROS message flow.

**No hardware required** -- the simulator replaces the physical IMU pads for testing.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== IMU Simulator Test ==="
echo ""
echo "A simulator window will open with clickable pad buttons."
echo ""
echo "Pad mapping for GUI navigation:"
echo "  LEFT pad  .... Previous option"
echo "  RIGHT pad .... Next option"
echo "  CENTRE pad ... Select / Enter"
echo "  HEAD pad ..... Go back"
echo ""
echo "Force levels (shown by colour):"
echo "  Light (green) | Medium (orange) | Hard (red)"
echo ""
echo "Click the pads and watch for nav events below."
echo "The monitor will run for 30 seconds, then stop automatically."
echo ""

# Start the simulator GUI in the background
python3 tools/imu_simulator.py &
SIM_PID=$!
echo "Simulator started (PID: $SIM_PID)"
sleep 2

echo ""
echo "--- Monitoring /boxbunny/imu/nav_event (30s timeout) ---"
echo "(Click pads in the simulator to see events appear here)"
echo ""
timeout 30 ros2 topic echo /boxbunny/imu/nav_event 2>/dev/null \
    || echo "(no events received -- click the pads in the simulator window)"

echo ""
echo "Stopping simulator..."
kill $SIM_PID 2>/dev/null
echo "Done."

---
## 25. CV Model Live Test
Run the action prediction model with the live camera feed. The window will show:
- YOLO pose skeleton overlay on your body
- Predicted punch type and confidence score
- Voxel activity heatmap

**Requires:** RealSense D435i camera + CV model files (see Section 11). Throw punches at the camera to see predictions update in real time. Press `q` or close the window to stop.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash

echo "=== Action Prediction Live Test ==="
echo ""
echo "A window will open showing the camera feed with overlays:"
echo "  - Pose skeleton (YOLO keypoints drawn on your body)"
echo "  - Predicted action label + confidence percentage"
echo "  - Voxel activity visualization"
echo ""
echo "Stand about 1.5-2m from the camera and throw punches!"
echo "Expected labels: jab, cross, hook_lead, hook_rear,"
echo "                 uppercut_lead, uppercut_rear, block, idle"
echo ""
echo "Press 'q' in the video window to stop."
echo ""

python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(failed -- is the D435i camera connected and are models downloaded?)"

---
## 26. Developer Mode Visual Test
Launch the GUI with the developer overlay enabled. The overlay appears in the bottom-right corner and shows:
- 4 pad rectangles (HEAD, LEFT, CENTRE, RIGHT) that flash in colour on impacts
- 2 arm rectangles (L ARM, R ARM) that flash on strikes
- Current CV prediction with confidence bar

Start the IMU simulator in another terminal (or run the simulator cell above first) to see pads light up when you click them.

In [ ]:
%%bash
set +e
source /opt/ros/humble/setup.bash && source install/setup.bash
export QT_QPA_PLATFORM=xcb

echo "=== Developer Mode Test ==="
echo ""
echo "The GUI will open with the developer overlay in the bottom-right corner."
echo "It shows:"
echo "  - 4 pad rectangles (HEAD, LEFT, CENTRE, RIGHT) that flash on impacts"
echo "  - 2 arm rectangles (L ARM, R ARM) that flash on strikes"
echo "  - Current CV prediction with confidence bar"
echo ""
echo "Starting IMU simulator in background for pad testing..."

# Start IMU simulator in background so you can click pads
python3 tools/imu_simulator.py &
SIM_PID=$!
echo "IMU Simulator PID: $SIM_PID"
sleep 1

echo ""
echo "Starting GUI with developer mode enabled..."
echo "Close the GUI window to stop."
echo ""

# Launch the GUI and immediately enable developer mode
python3 -c "
import sys, os
sys.path.insert(0, 'src/boxbunny_gui')
os.environ['QT_QPA_PLATFORM'] = 'xcb'

from boxbunny_gui.app import BoxBunnyApp
app = BoxBunnyApp()
app.set_developer_mode(True)
exit_code = app.run()
sys.exit(exit_code)
" 2>&1

# Cleanup
kill $SIM_PID 2>/dev/null
echo "Stopped IMU simulator."

---
## 27. View Demo User Dashboards
Renders styled profile cards for each demo user, pulling live data from their SQLite databases. This is a quick visual check that the seeded data looks correct.

**Prerequisite:** Run *Seed Demo Data* (Section 4) first.

In [ ]:
%run notebooks/scripts/view_user_dashboards.py

---
## 28. Benchmark Percentile Test
Test the population benchmark engine with each demo user's profile. The engine computes percentile rankings by comparing performance metrics against population norms segmented by age, gender, and skill level.

Results show where each user falls relative to the general population for their demographic bracket.

In [ ]:
%run notebooks/scripts/benchmark_test.py